In [12]:
import sys
sys.path.append('..')

In [13]:
from utils.prompts import render,list_prompts
from utils.router import pick_model
from utils.llm_client import LLMClient

In [14]:
print(list_prompts())

['zero_shot.v1', 'few_shot.v1']


In [15]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/sample_messages.txt")
output_path = Path("../output/classified_messages.xlsx")

# Read messages
with open(data_path, "r", encoding="utf-8") as file:
    messages_data = file.read()

# Messages are separated by blank lines
messages_list = [
    message.strip()
    for message in messages_data.split("\n\n")
    if message.strip()
]

print(f"Loaded {len(messages_list)} messages.")

Loaded 50 messages.


In [16]:
examples = """
Message:
"We are trapped on the roof with three children. Please send help immediately."

Output:
District: None | Intent: Rescue | Priority: High


Message:
"We urgently need drinking water and food supplies in Gampaha."

Output:
District: Gampaha | Intent: Supply | Priority: High


Message:
"Breaking News: Kelani River level at 9m."

Output:
District: Colombo | Intent: Info | Priority: Low


Message:
"Thanks to everyone helping the affected families."

Output:
District: None | Intent: Other | Priority: Low
"""

In [17]:
def classify_message(message, llm):
    prompt_text, spec = render(
        "few_shot.v1",
        role="crisis message classifier",
        examples=examples,
        query=message,
        constraints="""
Intent must be one of:
- Rescue
- Supply
- Info
- Other

Priority must be:
- High
- Low

If the district cannot be identified, use None.
""",
        format="District: [Name] | Intent: [Category] | Priority: [High/Low]"
    )

    response = llm.chat([
        {
            "role": "user",
            "content": prompt_text
        }
    ], temperature=spec.temperature, max_tokens=spec.max_tokens,
        task_type="classification")

    return response["text"].strip()

In [21]:
results = []

model = pick_model(
    provider="groq",
    technique="few_shot"
)
print(model)
llm = LLMClient("groq", model)

for index, message in enumerate(messages_list, start=1):

    classification = classify_message(message, llm)

    results.append({
        "Message": message,
        "Classification": classification
    })

    print(f"{index}. {message}")
    print(f"   {classification}")

openai/gpt-oss-20b
1. BREAKING: Water levels in Kelani River (Colombo) have reached 9.5 meters. Critical flood warning issued.
   District: Colombo | Intent: Info | Priority: High
2. SOS: 5 people trapped on a roof in Ja-Ela (Gampaha). Water rising fast. Need boat immediately.
   District: Gampaha | Intent: Rescue | Priority: High
3. Update: Kandy road cleared near Peradeniya. Traffic moving slowly. No victims reported.
   District: Kandy | Intent: Info | Priority: Low
4. Does anyone have extra dry rations for the camp in Gampaha?
   District: Gampaha | Intent: Supply | Priority: High
5. News just in: Kelani river water level is at 7ft.
   District: Colombo | Intent: Info | Priority: Low
6. My uncle is stuck in the tree (immediate danger).
   District: None | Intent: Rescue | Priority: High
7. Just saw on news that Gampaha town is flooded. Hope everyone is safe.
   District: Gampaha | Intent: Info | Priority: Low
8. We are trapped in the attic. 3 kids. Water is entering.
   District: N

In [22]:
output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.DataFrame(results)

df.to_excel(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Saved to: ..\output\classified_messages.xlsx
